# P1: DESMUNTANT LA MENTIDA VISUAL

## PREPROCESSAMENT DE DADES

In [1]:
%pip install pandas pycountry plotly "nbformat>=4.2.0"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\usuari\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import pycountry
import plotly.express as px
import plotly.graph_objects as go

In [3]:
pd.set_option("display.max_colwidth", None)

In [4]:
df = pd.read_csv("dades/dades_grafic.csv")
df.head()

,Indicator Name,Indicator Code,Country Name,Country Code,Year,Value,Disaggregation
0,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2025,63.674,"female, Modeled, 15+"
1,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2024,63.643,"female, Modeled, 15+"
2,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2023,63.593,"female, Modeled, 15+"
3,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2022,59.831,"female, Modeled, 15+"
4,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2021,59.454,"female, Modeled, 15+"


In [5]:
df.columns

Index(['Indicator Name', 'Indicator Code', 'Country Name', 'Country Code',
       'Year', 'Value', 'Disaggregation'],
      dtype='object')

El CSV original conté diferents indicadors per país, sexe i rang d'edat. Com que el gràfic que estic analitzant correspon al 2023, primer em quedo només amb les dades d'aquest any. A partir d'aquí exploraré els diferents indicadors per identificar quins són els que corresponen al gràfic original.

In [6]:
df_2023 = df[df["Year"] == 2023]
df_2023.shape

(2841, 7)

El primer que vull comprovar és l'estructura del dataset. Un mateix país apareix en diverses files perquè hi ha diferents indicadors, sexes i rangs d'edat. Per això comparo el nombre de files, el nombre d'entitats i els diferents Indicator Name disponibles.


In [7]:
rows = len(df_2023)
total_indicator_names = len(df_2023["Indicator Name"].unique())
print(f"Hi ha {total_indicator_names} Indicator Names diferents")
total_countries = len(df_2023["Country Name"].unique())
print(f"Hi ha {total_countries} països i {rows} files")


Hi ha 15 Indicator Names diferents
Hi ha 236 països i 2841 files


Tal com s'observa, hi ha moltes més files que països o regions, fet que confirma que cada entitat apareix repetida per als diferents indicadors. Per entendre millor aquesta estructura, agrupo les dades per Indicator Name.

També comprovo la correspondència entre Country Name i Country Code, per assegurar-me que no hi hagi països associats a més d'un codi abans de continuar amb el preprocessament.

In [8]:
if(len(df_2023["Country Name"].unique())==len(df_2023["Country Code"].unique())):
    print("Cada país té un codi associat.")
else:
    print("Hi ha discrepàncies entre el nombre de països i el nombre de codis")

Cada país té un codi associat.


Comprovo que realment un mateix country code només està associat a un país

In [9]:

paisos_codis = df.groupby("Country Name")["Country Code"].nunique()
if(max(paisos_codis)>1):
    print("Hi ha països associats a més d'un codi")
else:
    print("Cada país té un únic codi associat.")

Cada país té un únic codi associat.


In [10]:
indicador_names = df_2023.groupby("Indicator Name")
indicador_names.size()

Indicator Name
Labor force participation rate for ages 15-24, female (%) (modeled ILO estimate)                     231
Labor force participation rate for ages 15-24, female (%) (national estimate)                        127
Labor force participation rate for ages 15-24, male (%) (modeled ILO estimate)                       231
Labor force participation rate for ages 15-24, male (%) (national estimate)                          127
Labor force participation rate for ages 15-24, total (%) (modeled ILO estimate)                      231
Labor force participation rate for ages 15-24, total (%) (national estimate)                         127
Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)      231
Labor force participation rate, female (% of female population ages 15+) (national estimate)         127
Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)    231
Labor force participation rate, male (% 

En el gràfic posa escrit "Figures are rounded and based on the International Labour Organization's estimates as of 2023", per tant, treballaré amb aquells que indiquin "modeled ILO estimate". A partir d'aquí, cal averiguar quin és el rang d'edat (15-24, 15-64 o 15+) que es mostra al gràfic (ja que no està indicat).
Per comprovar quin rang d'edat s'utilitza en el gràfic, comparo les dades d'Afganistan per als diferents intervals disponibles. El gràfic mostra un 100% de participació laboral masculina i un 5% femenina, per tant, busco quin dels indicadors s'aproxima a aquests valors.

Tal i com s'observa, en el % d'homes en cap de les tres opcions coincideix amb el 100% que es mostra el gràfic, ja que els valors se situen entre el 68,598% i el 71,384%.
En canvi, en el % de dones, sí que hi ha el 5% en les entrades on l'edat és de 15-64 o de 15+. 
Per tant, miro pels 12 paísos del gràfic tots els valors, per veure si és un cas aïllat o és per tots els països.

In [11]:
#Tots els noms dels països del gràfic i del CSV no coincideixen, però amb agafant els nous noms ja s'agafen
paisos = [
    "Afghanistan",
    "Iraq",
    "Pakistan",
    "Oman",
    "Iran, Islamic Rep.",
    "Yemen, Rep.",
    "Egypt, Arab Rep.",
    "Algeria",
    "Syrian Arab Republic",
    "Jordan",
    "Morocco",
    "Saudi Arabia"
]
df_12 = df_2023[df_2023["Country Name"].isin(paisos)]
df_12 = df_12[df_12["Indicator Name"].str.contains("modeled ILO estimate")]
df_12_15 = df_12[df_12["Indicator Name"].str.contains("population ages 15-64",regex=False)]
df_12_15 = df_12_15.sort_values(["Country Name", "Indicator Name"])
df_12_15[["Country Name", "Indicator Name", "Value"]]


,Country Name,Indicator Name,Value
83544,Afghanistan,"Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",5.359
91954,Afghanistan,"Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)",71.384
100364,Afghanistan,"Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)",38.567
83616,Algeria,"Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",15.834
92026,Algeria,"Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)",73.378
100436,Algeria,"Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)",45.346
85308,"Egypt, Arab Rep.","Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",19.830
93718,"Egypt, Arab Rep.","Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)",74.050
102128,"Egypt, Arab Rep.","Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)",47.332
86352,"Iran, Islamic Rep.","Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",15.650


In [12]:
afghanistan=df_2023[df_2023["Country Name"] == "Afghanistan"]
afghanistan=afghanistan[afghanistan["Indicator Name"].str.contains("ILO estimate")]
afghanistan = afghanistan[["Indicator Name","Value"]]
afghanistan

,Indicator Name,Value
1694,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",5.145
15611,"Labor force participation rate, male (% of male population ages 15+) (modeled ILO estimate)",70.074
29527,"Labor force participation rate, total (% of total population ages 15+) (modeled ILO estimate)",37.588
43670,"Labor force participation rate for ages 15-24, female (%) (modeled ILO estimate)",6.013
56953,"Labor force participation rate for ages 15-24, male (%) (modeled ILO estimate)",68.598
70236,"Labor force participation rate for ages 15-24, total (%) (modeled ILO estimate)",38.006
83544,"Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",5.359
91954,"Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)",71.384
100364,"Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)",38.567


Tal com s'observa, els valors del CSV no coincideixen exactament amb els del gràfic original. He comparat els diferents rangs d'edat disponibles i, a partir d'aquest punt, treballaré amb la població de 15 a 64 anys i amb els indicadors modeled ILO estimate.
Amb aquestes dades reorganitzo el dataframe perquè cada país ocupi una fila i les taxes Female, Male i Total apareguin en columnes separades.

In [13]:
df_grafic = df_12_15.pivot(
    index="Country Name",
    columns="Indicator Name",
    values="Value"
)
df_grafic = df_grafic.rename(columns={
    "Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)": "Female",
    "Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)": "Male",
    "Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)": "Total"
})

df_grafic = df_grafic.reset_index()
df_grafic.columns.name = None
df_grafic

,Country Name,Female,Male,Total
0,Afghanistan,5.359,71.384,38.567
1,Algeria,15.834,73.378,45.346
2,"Egypt, Arab Rep.",19.830,74.050,47.332
3,"Iran, Islamic Rep.",15.650,73.892,45.302
4,Iraq,11.505,75.346,43.445
5,Jordan,17.057,66.084,42.620
6,Morocco,21.374,73.674,47.784
7,Oman,31.945,89.774,70.389
8,Pakistan,25.475,82.873,54.686
9,Saudi Arabia,35.617,84.185,66.724


Tal com s'observa, alguns valors coincideixen aproximadament, però en altres casos hi ha diferències importants. Per exemple, a Iraq el gràfic mostra un 92% per als homes, mentre que el CSV dona una taxa del 75,346%. En el cas d'Afghanistan, el gràfic mostra un 100%, mentre que al CSV la taxa és del 71,384%.
Com que no puc reproduir exactament tots els valors del gràfic original, a partir d'aquest punt utilitzaré com a referència les dades del CSV filtrades pel 2023, població de 15 a 64 anys i modeled ILO estimate. 

In [14]:
grafic_original = pd.DataFrame({
    "Country Name": paisos,
    "Grafic_Female": [5, 11, 25, 32, 14, 5, 17, 17, 14, 14, 20, 35],
    "Grafic_Male": [100, 92, 81, 88, 71, 60, 71, 67, 64, 63, 68, 80]
})
comparacio = df_grafic.merge(grafic_original, on="Country Name")
#Comparo els valors del gràfic amb els del dataset per veure'n les diferències.
comparacio["Dif_Female"] = abs(comparacio["Grafic_Female"] - comparacio["Female"])
comparacio["Dif_Male"] = abs(comparacio["Grafic_Male"] - comparacio["Male"])
comparacio = comparacio[["Country Name", "Female", "Grafic_Female", "Dif_Female", "Male", "Grafic_Male","Dif_Male"]]
comparacio


,Country Name,Female,Grafic_Female,Dif_Female,Male,Grafic_Male,Dif_Male
0,Afghanistan,5.359,5,0.359,71.384,100,28.616
1,Algeria,15.834,17,1.166,73.378,67,6.378
2,"Egypt, Arab Rep.",19.830,17,2.830,74.050,71,3.050
3,"Iran, Islamic Rep.",15.650,14,1.650,73.892,71,2.892
4,Iraq,11.505,11,0.505,75.346,92,16.654
5,Jordan,17.057,14,3.057,66.084,63,3.084
6,Morocco,21.374,20,1.374,73.674,68,5.674
7,Oman,31.945,32,0.055,89.774,88,1.774
8,Pakistan,25.475,25,0.475,82.873,81,1.873
9,Saudi Arabia,35.617,35,0.617,84.185,80,4.185


Considerant que els valors del gràfic no està clar d'on provenen, he volgut comprovar que els 12 països seleccionats siguin realment els 12 països amb  major diferència entre la participació laboral masculina i femenina segons les dades de l'ILO.
Analitzant els 12 països amb major diferència a nivell mundial, van aparèixer valors de regions i no de països, com per exemple Middle East & North Africa (IDA & IBRD), Middle East & North Africa (excluding high income), Arab World...
En python existeix una biblioteca per identificar països, per tant l'he aprofitat per seleccionar només aquelles files que corresponguessin a algun país segons el seu codi ISO (columna "Country Code" del dataframe)


In [15]:

#Creem una llista amb els codis ISO dels països
codis_paisos = {pais.alpha_3 for pais in pycountry.countries}
#Filtrem les dades del 2023 per a l'OIT i la població de 15-64 anys
df_mundial = df_2023[
    df_2023["Indicator Name"].str.contains("modeled ILO estimate", na=False)
    & df_2023["Indicator Name"].str.contains("population ages 15-64", regex=False, na=False)
].copy()
# Ens quedem només amb els països individuals
df_mundial = df_mundial[df_mundial["Country Code"].isin(codis_paisos)]

# Posem cada país en una fila i els indicadors en columnes
df_mundial = df_mundial.pivot(
    index=["Country Name", "Country Code"],
    columns="Indicator Name",
    values="Value"
).reset_index()

# Simplifiquem els noms de les columnes
df_mundial = df_mundial.rename(columns={
    "Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)": "Female",
    "Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)": "Male",
    "Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)": "Total"
})

# Traiem el nom de les columnes
df_mundial.columns.name = None

# Calculem la diferència entre homes i dones
df_mundial["Difference"] = df_mundial["Male"] - df_mundial["Female"]

# Ordenem els resultats i agafem els 12 primers
df_top12_mundial = df_mundial.sort_values(
    "Difference", ascending=False
).head(12)

# Mostrem les dades principals
df_top12_mundial[
    ["Country Name", "Country Code", "Female", "Male", "Difference"]
]

,Country Name,Country Code,Female,Male,Difference
0,Afghanistan,AFG,5.359,71.384,66.025
78,Iraq,IRQ,11.505,75.346,63.841
77,"Iran, Islamic Rep.",IRN,15.650,73.892,58.242
125,Oman,OMN,31.945,89.774,57.829
2,Algeria,DZA,15.834,73.378,57.544
180,"Yemen, Rep.",YEM,4.938,62.400,57.462
126,Pakistan,PAK,25.475,82.873,57.398
48,"Egypt, Arab Rep.",EGY,19.830,74.050,54.220
112,Morocco,MAR,21.374,73.674,52.300
159,Syrian Arab Republic,SYR,13.790,64.536,50.746


Tal com s’observa, els 12 països amb una diferència més gran coincideixen amb els que apareixen al gràfic original. Tot i això, aquesta coincidència no explica per què el gràfic decideix mostrar només aquests 12 països ni si existeix algun motiu concret que justifiqui aquest selecció.
Per aquest motiu, decideixo observar també les posicions següents del rànquing per comprovar si després del 12è país hi ha algun canvi o salt clar en els valors que puguin justificar aquesta selecció. El fet que els valors continuin disminuint de manera progressiva, implica que aquesta separació dels 12 països sigui artificial i es crei una separació que realment no existeix en les dades.

In [16]:
#Ordeno tots els països de més a menys diferènci
df_ranking = (
    df_mundial
    .sort_values("Difference", ascending=False)
    .reset_index(drop=True)
)

#Faig que l'índex coincideixi amb la posició del rànquing
df_ranking.index = df_ranking.index + 1

#Miro els 25 països amb una diferència més gran
df_ranking[
    ["Country Name", "Country Code", "Female", "Male", "Difference"]
].head(25)

,Country Name,Country Code,Female,Male,Difference
1,Afghanistan,AFG,5.359,71.384,66.025
2,Iraq,IRQ,11.505,75.346,63.841
3,"Iran, Islamic Rep.",IRN,15.650,73.892,58.242
4,Oman,OMN,31.945,89.774,57.829
5,Algeria,DZA,15.834,73.378,57.544
6,"Yemen, Rep.",YEM,4.938,62.400,57.462
7,Pakistan,PAK,25.475,82.873,57.398
8,"Egypt, Arab Rep.",EGY,19.830,74.050,54.220
9,Morocco,MAR,21.374,73.674,52.300
10,Syrian Arab Republic,SYR,13.790,64.536,50.746


El 12è país és Saudi Arabia, amb una diferència de 48,568 punts percentuals, mentre que el 13è és India, amb 47,674. La diferència entre tots dos és només de 0,894 punts percentuals i, si es miren també les posicions següents, no s'observa cap salt clar que justifiqui separar els 12 primers de la resta.
Per tant, considero que destacar únicament aquests 12 països no és per cap motiu concret. Aquesta separació podria donar la sensació que existeix un grup clarament diferenciat de països amb una desigualtat especialment elevada, quan en realitat els valors disminueixen de manera bastant progressiva. Per aquest motiu, decideixo treballar amb tots els països disponibles i representar la distribució a escala mundial, de manera que es pugui observar millor la continuïtat de les diferències i evitar crear agrupacions visuals que no estan realment justificades per les dades.
El següent dubte que em plantejo és si hi ha algun país on la taxa de participació laboral femenina sigui superior a la masculina, per tant miro el valor mínim del dataframe. Com que aquest és negatiu, implica que hi ha països on la participació laboral femenina és superior a la masculina. 

In [17]:
#Miro com es distribueix la diferència entre tots els països
df_mundial["Difference"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
)

count    183.000000
mean      17.805415
std       14.798723
min       -3.351000
25%        7.439500
50%       12.470000
75%       25.625000
90%       38.675600
95%       50.574100
max       66.025000
Name: Difference, dtype: float64

Com que he observat que hi havia un valor mínim negatiu, he volgut explorar el dataframe per identificar en quins països la taxa de participació laboral femenina és superior a la masculina. Amb el codi següent s’observa que només hi ha dos casos: Burundi, amb una diferència de -3,351 punts percentuals, i Moldova, amb una diferència de -0,421 punts percentuals. Això indica que, en aquests dos països, la taxa femenina supera lleugerament la masculina.


In [18]:
#Miro els països on la taxa femenina és superior a la masculina
df_mundial[df_mundial["Difference"] < 0][["Country Name", "Female", "Male", "Difference"]].sort_values("Difference")

,Country Name,Female,Male,Difference
25,Burundi,81.292,77.941,-3.351
109,Moldova,72.727,72.306,-0.421


Fins ara he comparat les taxes de participació laboral masculina i femenina mitjançant Male - Female. Aquesta diferència s'expressa en punts percentuals i permet comparar directament les dues taxes.
És important tenir en compte que cada percentatge es calcula respecte al seu propi sexe. La taxa femenina indica quin percentatge de totes les dones de 15 a 64 anys forma part de la força laboral, mentre que la taxa masculina indica el percentatge equivalent entre els homes de 15 a 64 anys. Per tant, aquests valors no representen quin percentatge de la força laboral total són dones o homes.
També he provat d'expressar la diferència de manera relativa:Relative Difference = ((Male - Female) / Female) * 100
Tot i això, quan la taxa femenina és molt baixa apareixen valors molt elevats. Per exemple, Afghanistan arriba al 1232,04% i Yemen al 1163,67%. Encara que el càlcul sigui correcte, considero que aquesta manera d'expressar la diferència és menys intuïtiva i podria exagerar visualment alguns casos.
També vaig explorar la possibilitat d'utilitzar la població masculina i femenina de cada país per estimar quin percentatge de la força laboral correspon a cada sexe. Aquesta opció em va semblar interessant perquè el títol del gràfic original, *“Top 12 countries where men outnumber women in the workforce”*, pot fer pensar que s'està comparant quants homes i quantes dones formen part de la força laboral.
Tot i això, les dades representades al gràfic original no mostren aquesta composició. El percentatge femení es calcula sobre el total de dones de 15 a 64 anys i el masculí sobre el total d'homes de la mateixa edat. Per tant, el que realment s'està comparant són les taxes de participació laboral dels dos sexes.
Calcular la composició de la força laboral a partir de les dades de població permetria respondre més literalment a la idea suggerida pel títol original, però implicaria canviar la pregunta que responen les dades i introduir una nova font d'informació.
Per aquest motiu, finalment mantinc la diferència en punts percentuals com a mesura principal, ja que compara directament les dues taxes representades al gràfic original. Com que una mateixa diferència pot correspondre a nivells de participació diferents, al hover mostro també les taxes femenina i masculina de cada país. A més, al títol i al hover intento deixar clar respecte de quin grup es calcula cada percentatge per evitar la mateixa ambigüitat del gràfic original.

## GRÀFIC MILLORAT

Al principi vaig plantejar-me destacar només els 12 països amb una diferència més gran i deixar la resta del mapa en un color neutre. Tot i això, després d’explorar millor les dades vaig veure que no hi havia una separació clara entre el país 12 i els següents. Per exemple, Aràbia Saudita té una diferència de 48,568 pp i l’Índia, que ocupa la posició 13, de 47,674 pp.
Per això, vaig decidir representar tots els països segons la diferència entre la taxa de participació laboral masculina i femenina. D’aquesta manera no creo un tall artificial entre els 12 primers i la resta, i es pot veure millor com es distribueixen realment les diferències a escala mundial.
A més, explorant les dades vaig veure que hi ha dos països, Burundi i Moldova, on la taxa femenina és lleugerament superior a la masculina. Això també m’ha fet replantejar l’escala de colors per poder representar tant la magnitud com la direcció de la diferència.


Opcions que he considerat pel color:
1. Escala de colors centrada en 0: rosa si Female > Male, neutre a 0 i blau si Male > Female. El problema és que Burundi i Moldova tenen valors molt propers a 0 i gairebé no es veuria el rosa.
2. Escala de colors + marcador extra: mantenir l’escala anterior però marcar BDI i MDA amb una vora o codi rosa perquè es vegin sense exagerar la diferència.
3. Dues escales separades: una pels valors positius i una pels negatius. Ho he descartat perquè podria fer semblar que les intensitats són comparables quan no ho són.

Finalment, he decidit mantenir una escala de colors centrada en 0, sense forçar que els valors negatius es vegin més del que realment representen. El fet que gairebé no aparegui rosa al mapa també aporta informació, ja que mostra que els casos on la taxa femenina és superior són molt pocs i amb diferències molt petites.
Per tant, el rosa representa Female > Male, el color neutre valors propers a 0 i el blau Male > Female. Com més intens és el color, més gran és la diferència. D’aquesta manera puc representar alhora la direcció i la magnitud sense exagerar cap dels dos casos excepcionals.


A nivell de colors, he près les següents decisions:
1. He decidit diferenciar els països sense dades amb un color gris. D’aquesta manera no es poden confondre amb els països que sí tenen dades però presenten una diferència propera a 0.
2. He mantingut el 0 com a punt neutre de l’escala de colors. A partir d’aquest punt, els tons blaus representen una taxa masculina superior i els tons roses una taxa femenina superior.
3. He decidit no intensificar artificialment els tons roses. Com que Burundi i Moldova tenen diferències negatives molt petites, és normal que el rosa gairebé no sigui visible al mapa. Fer-lo més intens podria donar més importància a aquesta diferència de la que realment té.
4. He utilitzat diferents intensitats de blau perquè es pugui distingir entre els països amb diferències més petites i els que presenten diferències més grans.


Com que Burundi i Moldova tenen diferències negatives molt petites, al mapa gairebé no es veu el rosa. Per això he utilitzat també el hover per reforçar aquesta informació: el fons és rosa quan Female > Male i blau quan Male > Female. Així no exagero la diferència al mapa, però quan l’usuari passa per sobre del país pot veure clarament en quina direcció va i consultar els valors exactes.

In [ ]:
import plotly.graph_objects as go

#Faig una còpia del dataframe original per usar-lo pel mapa
df_mapa = df_mundial.copy()

#Poso els noms dels països en majúscules pel hover
df_mapa["Country Name"] = df_mapa["Country Name"].str.upper()

#Assigno un color al hover segons quina taxa de participació és superior
df_mapa["HoverColor"] = df_mapa["Difference"].apply(
    lambda x: "#D98EAA" if x < 0 else "#8FB9D9" if x > 0 else "#CFCFCF"
)
#Creem un mapa buit on després afegirem les capes
fig = go.Figure()

#Afegir tots els països amb dades
fig.add_trace(
    go.Choropleth(
        locations=df_mapa["Country Code"],     #Indicar quins són els països
        z=df_mapa["Difference"],               #El color dependrà de la seva diferència

        #Faig servir una escala de colors centrada en 0 per diferenciar en quina direcció va la diferència
        colorscale=[
            [0.0, "#C94F7C"],    #Taxa femenina superior
            [0.45, "#E8C4D1"],   #Valors negatius propers a 0
            [0.5, "#F5F5F5"],    #Taxes similars
            [0.60, "#C5D8EA"],   #Diferència masculina baixa
            [0.75, "#7FA7CC"],   #Diferència masculina mitjana
            [1.0, "#2F5F8F"]     #Diferència masculina alta
        ],
        zmid=0,

       customdata=df_mapa[[
        "Country Name",
        "Female",
        "Male",
        "Difference"]
        ], #Guardem dades extra per mostrar en passar per sobre
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "<b>Participation rates</b><br>"
            "Women: %{customdata[1]:.1f}% of women aged 15–64<br>"
            "Men: %{customdata[2]:.1f}% of men aged 15–64<br>"
            "<b>Difference: %{customdata[3]:.1f} pp</b>"
            "<extra></extra>"
        ),

        #Personalitzo el hover perquè sigui més visual
        hoverlabel=dict(
            bgcolor=df_mapa["HoverColor"].tolist(),
            bordercolor="#FFFFFF",
            align="left"
        ),

        colorbar=dict(
            title="",
            x=1.095,
            len=0.5
        )
    )
)

fig.add_annotation(
    text="Male-female<br>difference",
    x=1.13,
    y=0.82,
    xref="paper",
    yref="paper",
    xanchor="center",
    showarrow=False,
    font=dict(
        family="Times New Roman",
        size=14
    )
)

#Afegeixo una indicació pels països que no tenen dades
fig.add_annotation(
    text="■ No data available",
    x=1.13,
    y=0.19,
    xref="paper",
    yref="paper",
    xanchor="center",
    showarrow=False,
    font=dict(
        family="Times New Roman",
        size=15,
        color="#777777"
    )
)


#Afegeixo el meu tìtol
fig.update_layout(
    width=1100,   #Amplada del gràfic
    height=700,   #Alçada del gràfic
    margin=dict(r=200),

    #Diferencio els països sense dades amb un color gris
    geo=dict(
        showland=True,
        landcolor="#C8C8C8"
    ),

    #Poso la mateixa font per tots els elements del hover
    hoverlabel=dict(
        font=dict(
            family="Times New Roman",
            size=14,
            color="#2F2F2F"
        )
    ),

    title=dict(
        text=(
        "Labour Force Participation by Sex Worldwide"
        "<br><sup>"
        "Participation rate within each sex, ages 15-64, 2023. Modeled ILO estimate. "
        "Color = male - female rate (pp)."
        "</sup>"
        ),
        x=0.5,  #Centrem el títol
        y=0.90, #Baixem una mica el bloc de text
        yanchor="top",
        xanchor="center",
        font=dict(
            family="Times New Roman",
            size=30
        )
    )
)

fig.show()